# Anomaly Detection: Which Plots/Pixels Are Outliers?

Two independent methods, cross-checked against each other, applied to
all 3 HF365 indices (42 plots each) and both MODIS variables (5 pixels
each). A plot/pixel is only reported as a confirmed outlier if **both**
methods agree.


In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_style("whitegrid")

# vvv EDIT THESE to match your folders vvv
HF365_DIR = "."
MODIS_DIR = "modis_downloads"
# ^^^ EDIT THESE to match your folders vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}


## 1. Load HF365 and MODIS data

In [ ]:
hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")


In [ ]:
FILE_PATTERN_TEMPLATE = r"MCD15A2H\.061_{layer}_(\d{{8}})T\d+_aid(\d+)"
SCALE = {"Lai_500m": 0.1, "Fpar_500m": 0.01}
FILL_VALUES = {249, 250, 251, 252, 253, 254, 255}
all_modis_files = glob.glob(os.path.join(MODIS_DIR, "**", "*.tif"), recursive=True)

def load_modis_layer(layer_name):
    pattern = re.compile(FILE_PATTERN_TEMPLATE.format(layer=re.escape(layer_name)))
    files = [p for p in all_modis_files if layer_name in os.path.basename(p)]
    records = []
    for path in files:
        m = pattern.search(os.path.basename(path))
        if not m:
            continue
        date_str, pixel_num = m.groups()
        with rasterio.open(path) as src:
            arr = src.read(1)
            raw = arr[1, 1] if arr.shape == (2, 2) else np.nan
        value = np.nan if raw in FILL_VALUES else raw * SCALE[layer_name]
        records.append({"date": pd.to_datetime(date_str, format="%Y%m%d"),
                         "pixel": f"pixel_{int(pixel_num)}", "value": value})
    return pd.DataFrame(records).pivot_table(index="date", columns="pixel", values="value", aggfunc="first").sort_index()

modis_wide = {v: load_modis_layer(v) for v in ["Lai_500m", "Fpar_500m"]}
for name, df in modis_wide.items():
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} pixels")

all_datasets = {**hf365_wide, **modis_wide}


## 2. Method A: raw time-series centroid distance

Each plot/pixel's own time series compared directly to the group
centroid (mean series). Flags whichever ones sit furthest from that
centroid.

In [ ]:
def centroid_distance_raw(df, top_pct=0.9):
    centroid = df.mean(axis=1)
    dist = df.apply(lambda col: np.sqrt(((col - centroid)**2).mean(skipna=True)))
    threshold = dist.quantile(top_pct)
    return pd.DataFrame({"distance": dist, "flagged": dist > threshold}).sort_values("distance", ascending=False)

method_a = {name: centroid_distance_raw(df) for name, df in all_datasets.items()}
for name, res in method_a.items():
    print(f"{name}: {res['flagged'].sum()} flagged of {len(res)}")


## 3. Method B: behavioral-summary centroid distance (cross-check)

Each plot/pixel reduced to summary features (mean, std, seasonal
amplitude) first, then compared to the centroid in that feature space
-- a structurally different method from Method A, so agreement between
the two is a much stronger signal than either alone.

In [ ]:
def build_behavior_features(df):
    feats = pd.DataFrame(index=df.columns)
    feats["mean"] = df.mean(axis=0)
    feats["std"] = df.std(axis=0)
    monthly = df.groupby(df.index.month).mean()
    feats["seasonal_amplitude"] = monthly.max(axis=0) - monthly.min(axis=0)
    return feats.dropna()

def centroid_distance_features(df, top_pct=0.9):
    feats = build_behavior_features(df)
    X = StandardScaler().fit_transform(feats)
    centroid = X.mean(axis=0)
    dist = pd.Series(np.linalg.norm(X - centroid, axis=1), index=feats.index)
    threshold = dist.quantile(top_pct)
    return pd.DataFrame({"distance": dist, "flagged": dist > threshold}).sort_values("distance", ascending=False), X, feats

method_b = {}
method_b_X = {}
for name, df in all_datasets.items():
    method_b[name], method_b_X[name], _ = centroid_distance_features(df)
    print(f"{name}: {method_b[name]['flagged'].sum()} flagged of {len(method_b[name])}")


## 4. Agreement check -- flagged by BOTH methods

In [ ]:
agreement = {}
for name in all_datasets:
    flagged_a = set(method_a[name][method_a[name]["flagged"]].index)
    flagged_b = set(method_b[name][method_b[name]["flagged"]].index)
    agreement[name] = sorted(flagged_a & flagged_b)
    print(f"{name} -- flagged by Method A: {sorted(flagged_a)}")
    print(f"{name} -- flagged by Method B: {sorted(flagged_b)}")
    print(f"{name} -- agree on: {agreement[name]}\n")


## 5. Final report table

In [ ]:
report_rows = []
for name in all_datasets:
    for item in method_a[name].index:
        report_rows.append(dict(
            dataset=name, item=item,
            raw_distance=method_a[name].loc[item, "distance"],
            feature_distance=method_b[name].loc[item, "distance"] if item in method_b[name].index else np.nan,
            flagged_by_both=item in agreement[name],
        ))
report_df = pd.DataFrame(report_rows).sort_values(["dataset","raw_distance"], ascending=[True,False])
report_df[report_df["flagged_by_both"]]


## 6. PCA visualization -- behavioral feature space

2D projection of the behavioral feature space per dataset, with agreed
outliers circled.

In [ ]:
def plot_pca(name):
    feats = build_behavior_features(all_datasets[name])
    X = method_b_X[name]
    if X.shape[1] < 2:
        print(f"{name}: not enough features for a 2D PCA plot"); return
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(X)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.scatter(X2[:,0], X2[:,1], s=50, color="steelblue")
    flagged_mask = feats.index.isin(agreement[name])
    ax.scatter(X2[flagged_mask,0], X2[flagged_mask,1], facecolors="none", edgecolors="red", s=180, linewidths=2)
    for i, label in enumerate(feats.index):
        ax.annotate(label, (X2[i,0], X2[i,1]), fontsize=7, alpha=0.7)
    ax.set_title(f"Behavioral feature space (PCA) -- {name}")
    plt.tight_layout(); plt.show()

for name in all_datasets:
    plot_pca(name)


## 7. Peer correlation cross-check

A third, independent signal: does a plot/pixel with unusually *low*
average correlation to its peers line up with the plots/pixels already
flagged above?

In [ ]:
def peer_correlation_summary(df):
    corr = df.corr()
    mask = np.eye(len(corr), dtype=bool)
    corr_masked = corr.mask(mask)
    return corr_masked.mean(axis=1).sort_values()

for name, df in all_datasets.items():
    peer_corr = peer_correlation_summary(df)
    print(f"{name} -- lowest average peer correlation:")
    print(peer_corr.head(5).round(3))
    print()


## 8. Mahalanobis vs Euclidean distance-to-centroid

Section 3's behavioral distance used **Euclidean** distance in the
standardized feature space -- which implicitly assumes each feature
is independent and equally scaled. **Mahalanobis distance** additionally
accounts for how the features actually *covary* with each other (e.g.
if `mean` and `seasonal_amplitude` are correlated across plots,
Euclidean distance double-counts that shared signal; Mahalanobis
doesn't). Comparing the two directly shows how much this matters for
this dataset.

In [ ]:
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv, LinAlgError

def compute_distance_pair(name):
    feats = build_behavior_features(all_datasets[name])
    X = method_b_X[name]  # already-standardized feature matrix from Section 3
    centroid = X.mean(axis=0)

    euclid = np.linalg.norm(X - centroid, axis=1)

    try:
        cov = np.cov(X, rowvar=False)
        cov_inv = inv(cov) if X.shape[1] > 1 else np.array([[1.0 / cov]])
        mahal = np.array([mahalanobis(row, centroid, cov_inv) for row in X])
    except LinAlgError:
        mahal = np.full(len(X), np.nan)  # singular covariance (e.g. too few observations relative to features)

    return pd.DataFrame({"euclidean": euclid, "mahalanobis": mahal}, index=feats.index)

distance_pairs = {name: compute_distance_pair(name) for name in all_datasets}
for name, d in distance_pairs.items():
    r = d["euclidean"].corr(d["mahalanobis"])
    print(f"{name}: Euclidean vs Mahalanobis correlation = {r:.3f}" if pd.notna(r) else
          f"{name}: Mahalanobis undefined (singular covariance -- too few observations for the feature count)")


### Scatter: Euclidean vs Mahalanobis, per dataset

Points that sit far from the diagonal are exactly where the two
methods *disagree* about how anomalous something is -- worth a closer
look at those specifically, since that's where accounting for
feature covariance actually changes the conclusion.

In [ ]:
n_datasets = len(distance_pairs)
fig, axes = plt.subplots(1, n_datasets, figsize=(4.5*n_datasets, 4.5))
for ax, (name, d) in zip(axes, distance_pairs.items()):
    if d["mahalanobis"].isna().all():
        ax.text(0.5, 0.5, "Mahalanobis undefined\n(singular covariance)", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(name)
        continue
    ax.scatter(d["euclidean"], d["mahalanobis"], s=40, color="steelblue")
    for item in d.index:
        ax.annotate(item, (d.loc[item,"euclidean"], d.loc[item,"mahalanobis"]), fontsize=6, alpha=0.7)
    ax.set_xlabel("Euclidean distance to centroid")
    ax.set_ylabel("Mahalanobis distance to centroid")
    ax.set_title(name)
plt.tight_layout()
plt.show()


### Table: both distances side by side, sorted by Mahalanobis

In [ ]:
for name, d in distance_pairs.items():
    print(f"\n{name} -- top 5 by Mahalanobis distance:")
    print(d.sort_values("mahalanobis", ascending=False).head(5).round(3))


## Summary

- A plot/pixel appearing in Section 5's table (flagged by both methods)
  is the strongest evidence of genuine anomalous behavior in this
  analysis.
- Section 7 (peer correlation) is a third, independent cross-check --
  if a name shows up there too, that's further corroboration.
- Section 8 adds a fourth lens: **Mahalanobis distance**, which accounts
  for feature covariance that plain Euclidean distance ignores. Where
  the two disagree (off-diagonal points in the scatter plots) is worth
  a specific look -- that's exactly where "how anomalous is this"
  depends on which distance measure you trust.
- **Known limitation:** for MODIS (only 5 pixels), a percentile-based
  threshold has very little room to discriminate, and Mahalanobis
  distance itself can become undefined (singular covariance) when the
  number of observations is close to the number of features. Treat
  pixel-level flags with more caution than plot-level flags (n=42) for
  this reason.
